In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
# load data set
train_path = r'E:\machine_learning_practice\practice\machine_learning_handson\data\train.csv'
test_path = r'E:\machine_learning_practice\practice\machine_learning_handson\data\test.csv'
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

In [3]:
train.shape

(1460, 81)

In [4]:
test.shape

(1459, 80)

In [5]:
X_train = train.drop(columns="SalePrice",axis = 1)
y_train = train["SalePrice"]
X_test = test.copy()
print("X_train shape",X_train.shape)
print("y_train shape",y_train.shape)
print("X_test shape",X_test.shape)

X_train shape (1460, 80)
y_train shape (1460,)
X_test shape (1459, 80)


# Missing value Imputation

In [6]:
isnull_sum = X_train.isnull().sum()
isnull_sum

Id                 0
MSSubClass         0
MSZoning           0
LotFrontage      259
LotArea            0
                ... 
MiscVal            0
MoSold             0
YrSold             0
SaleType           0
SaleCondition      0
Length: 80, dtype: int64

# finding the numerical variable which have missing value

In [7]:
num_vars = X_train.select_dtypes(include = ["int64","float64"]).columns
num_vars_miss = [var for var in num_vars if isnull_sum[var]>0]
num_vars_miss

['LotFrontage', 'MasVnrArea', 'GarageYrBlt']

# finding the categorical variable which have missing value

In [8]:
cat_vars = X_train.select_dtypes(include = ["object"]).columns
cat_vars_miss = [var for var in cat_vars if isnull_sum[var]>0]
cat_vars_miss

['Alley',
 'MasVnrType',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Electrical',
 'FireplaceQu',
 'GarageType',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PoolQC',
 'Fence',
 'MiscFeature']

In [9]:
num_var_mean =["LotFrontage"]
num_var_median =['MasVnrArea', 'GarageYrBlt']
cat_vars_mode =['Alley',
 'MasVnrType',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Electrical',
 'FireplaceQu',]
cat_vars_missing = ['GarageType',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PoolQC',
 'Fence',
 'MiscFeature']

In [10]:
num_var_mean_imputer = Pipeline(steps=[('imputer', SimpleImputer(strategy="mean"))])
num_var_median_imputer = Pipeline(steps=[('imputer', SimpleImputer(strategy="median"))])
cat_vars_mode_imputer = Pipeline(steps=[('imputer', SimpleImputer(strategy="most_frequent"))])
cat_vars_missing_imputer = Pipeline(steps=[('imputer', SimpleImputer(strategy="constant",fill_value="Missing"))])

In [11]:
preprocessor = ColumnTransformer(transformers=[
    ('mean_imputer',num_var_mean_imputer,num_var_mean),
    ('median_imputer',num_var_median_imputer,num_var_median ),
    ('mode_imputer',cat_vars_mode_imputer,cat_vars_mode),
    ('missing_imputer',cat_vars_missing_imputer,cat_vars_missing),
], remainder='passthrough')

In [12]:
preprocessor.fit(X_train)

C:\Users\Jeet\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


ColumnTransformer(remainder='passthrough',
                  transformers=[('mean_imputer',
                                 Pipeline(steps=[('imputer', SimpleImputer())]),
                                 ['LotFrontage']),
                                ('median_imputer',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['MasVnrArea', 'GarageYrBlt']),
                                ('mode_imputer',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent'))]),
                                 ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond',
                                  'BsmtExposure', 'BsmtFinType1',
                                  'BsmtFinType2', 'Electrical',
                                  'FireplaceQu']),
                                ('missing_imputer',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant'))]),
                                 ['GarageType', 'GarageFinish', 'GarageQual',
                                  'GarageCond', 'PoolQC', 'Fence',
                                  'MiscFeature'])])

In [13]:
preprocessor.transform

<bound method ColumnTransformer.transform of ColumnTransformer(remainder='passthrough',
                  transformers=[('mean_imputer',
                                 Pipeline(steps=[('imputer', SimpleImputer())]),
                                 ['LotFrontage']),
                                ('median_imputer',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['MasVnrArea', 'GarageYrBlt']),
                                ('mode_imputer',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent'))]),
                                 ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond',
                                  'BsmtExposure', 'BsmtFinType1',
                                  'BsmtFinType2', 'Electrical',
                                  'FireplaceQu']),
 

In [14]:
preprocessor.named_transformers_['mean_imputer'].named_steps['imputer'].statistics_

array([70.04995837])

In [15]:
train["LotFrontage"].mean()

70.04995836802665

In [16]:
preprocessor.named_transformers_['mode_imputer'].named_steps['imputer'].statistics_

array(['Grvl', 'BrkFace', 'TA', 'TA', 'No', 'Unf', 'Unf', 'SBrkr', 'Gd'],
      dtype=object)

In [17]:
X_train_clean = preprocessor.transform(X_train)
X_test_clean = preprocessor.transform(X_test)


In [18]:
X_train_clean

array([[65.0, 196.0, 2003.0, ..., 2008, 'WD', 'Normal'],
       [80.0, 0.0, 1976.0, ..., 2007, 'WD', 'Normal'],
       [68.0, 162.0, 2001.0, ..., 2008, 'WD', 'Normal'],
       ...,
       [66.0, 0.0, 1941.0, ..., 2010, 'WD', 'Normal'],
       [68.0, 0.0, 1950.0, ..., 2010, 'WD', 'Normal'],
       [75.0, 0.0, 1965.0, ..., 2008, 'WD', 'Normal']], dtype=object)

In [19]:
X_test_clean

array([[80.0, 0.0, 1961.0, ..., 2010, 'WD', 'Normal'],
       [81.0, 108.0, 1958.0, ..., 2010, 'WD', 'Normal'],
       [74.0, 0.0, 1997.0, ..., 2010, 'WD', 'Normal'],
       ...,
       [160.0, 0.0, 1960.0, ..., 2006, 'WD', 'Abnorml'],
       [62.0, 0.0, 1980.0, ..., 2006, 'WD', 'Normal'],
       [74.0, 94.0, 1993.0, ..., 2006, 'WD', 'Normal']], dtype=object)

In [20]:
preprocessor.transformers_

[('mean_imputer',
  Pipeline(steps=[('imputer', SimpleImputer())]),
  ['LotFrontage']),
 ('median_imputer',
  Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]),
  ['MasVnrArea', 'GarageYrBlt']),
 ('mode_imputer',
  Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent'))]),
  ['Alley',
   'MasVnrType',
   'BsmtQual',
   'BsmtCond',
   'BsmtExposure',
   'BsmtFinType1',
   'BsmtFinType2',
   'Electrical',
   'FireplaceQu']),
 ('missing_imputer',
  Pipeline(steps=[('imputer',
                   SimpleImputer(fill_value='Missing', strategy='constant'))]),
  ['GarageType',
   'GarageFinish',
   'GarageQual',
   'GarageCond',
   'PoolQC',
   'Fence',
   'MiscFeature']),
 ('remainder',
  FunctionTransformer(accept_sparse=True, check_inverse=False,
                      feature_names_out='one-to-one'),
  [0, 1, 2, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 27, 28, 29, 34, 36, 37, 38, 39, 40, 41, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 

In [23]:
X_train_clean_mis_var = pd.DataFrame(X_train_clean,columns=X_train.columns,index = X_train.index)
X_train_clean_mis_var

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,65.0,196.0,2003.0,Grvl,BrkFace,Gd,TA,No,GLQ,Unf,...,61,0,0,0,0,0,2,2008,WD,Normal
1,80.0,0.0,1976.0,Grvl,BrkFace,Gd,TA,Gd,ALQ,Unf,...,0,0,0,0,0,0,5,2007,WD,Normal
2,68.0,162.0,2001.0,Grvl,BrkFace,Gd,TA,Mn,GLQ,Unf,...,42,0,0,0,0,0,9,2008,WD,Normal
3,60.0,0.0,1998.0,Grvl,BrkFace,TA,Gd,No,ALQ,Unf,...,35,272,0,0,0,0,2,2006,WD,Abnorml
4,84.0,350.0,2000.0,Grvl,BrkFace,Gd,TA,Av,GLQ,Unf,...,84,0,0,0,0,0,12,2008,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,62.0,0.0,1999.0,Grvl,BrkFace,Gd,TA,No,Unf,Unf,...,40,0,0,0,0,0,8,2007,WD,Normal
1456,85.0,119.0,1978.0,Grvl,Stone,Gd,TA,No,ALQ,Rec,...,0,0,0,0,0,0,2,2010,WD,Normal
1457,66.0,0.0,1941.0,Grvl,BrkFace,TA,Gd,No,GLQ,Unf,...,60,0,0,0,0,2500,5,2010,WD,Normal
1458,68.0,0.0,1950.0,Grvl,BrkFace,TA,TA,Mn,GLQ,Rec,...,0,112,0,0,0,0,4,2010,WD,Normal


In [24]:
X_train_clean_mis_var.isnull().sum().sum()

0

In [ ]:
train["Alley"].value_counts()

In [ ]:
X_train_clean_mis_var["Alley"].value_counts()

In [ ]:
X_train_clean_mis_var['MiscFeature'].value_counts()